# 🔬 Arogya Sahayak — Deep Learning Scanner 08: Skin Disease Classification
### ISIC HAM10000 Dermoscopic Lesion Classification, Dermal Grad-CAM & TensorFlow.js Edge Export

**Project:** Arogya Sahayak — Offline-First Multilingual Rural Telemedicine Assistant  
**Institution:** ABIT Cuttack, Affiliated to BPUT Odisha  
**Student Developers:** Rahul Prasad Das (2301206094), Purnachandra Swain (2301206093), Rohit Kumar Thakur (2301206103), Ritik Kumar Swain (2301206102)  
**Project Guide:** Prof. Rajesh Kumar Sahoo, Assistant Professor, Dept. of CSE  
**Constraint:** 100% Free Tiers (Google Colab T4 GPU, Kaggle Datasets, Zero Cost)  

---

### 🎯 Clinical Objectives:
1. **Dermoscopic Neoplasm Screening:** Classify dermatoscopic lesion imagery across the 7 gold-standard ISIC HAM10000 diagnostic categories:
   - **`0: Melanocytic Nevus (Benign Mole - nv)`** — Symmetrical, regular pigment network, uniform brown pigmentation, benign melanocytic proliferation.
   - **`1: Melanoma (Malignant Neoplasm - mel)`** — Asymmetry, irregular borders, multiple color variegations (black, brown, white, blue), atypical pigment network, regression structures.
   - **`2: Benign Keratosis (Seborrheic Plaque - bkl)`** — Milia-like cysts, comedo-like openings, crypts, fingerprinted/cerebriform pattern, solar lentigines.
   - **`3: Basal Cell Carcinoma (BCC - bcc)`** — Arborizing telangiectasia, blue-gray ovoid nests/globules, ulceration, spoke-wheel structures.
   - **`4: Actinic Keratosis / Bowen's (Pre-Cancerous - akiec)`** — Erythematous background, keratotic scale, surface crusting, strawberry pattern.
   - **`5: Vascular Lesion (Angioma - vasc)`** — Red, purple, or blue-black lacunae/clods, absence of pigment network.
   - **`6: Dermatofibroma (Benign - df)`** — Central white patch / scar-like area surrounded by a delicate delicate peripheral pigment network.
2. **MobileNetV2 Deep Transfer Learning:** Train on the 10,015 dermatoscopic images of the HAM10000 benchmark with progressive unfreezing.
3. **Clinical Evaluation:** Quantify diagnostic Sensitivity (Recall for Melanoma and Carcinomas), Specificity, Macro/Weighted F1-scores, and Confusion Matrices.
4. **Dermal Grad-CAM Explainability:** Extract authentic partial derivatives from the final convolutional layer (`Conv_1`) to verify that the network focuses on pigment network asymmetry and border characteristics rather than surrounding healthy skin, hair, or vignetting.
5. **Edge Deployment:** Export to sub-10 MB TensorFlow.js graph weights for zero-latency client-side mobile dermatological screening.

---
## 1. Environment Setup & Hardware Acceleration
Verify GPU accelerator availability (NVIDIA Tesla T4 on free Colab) and import deep learning libraries.

In [ ]:
# Verify GPU accelerator
!nvidia-smi

# Install dependencies (quiet mode)
!pip install -q kagglehub tensorflowjs scikit-learn seaborn matplotlib pandas pillow

import os
import glob
import json
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Devices: {tf.config.list_physical_devices('GPU')}")

---
## 2. Dataset Acquisition via Kaggle API
We use the gold-standard Skin Cancer MNIST: HAM10000 dataset (`kmader/skin-cancer-mnist-ham10000`), containing 10,015 dermatoscopic images verified by histopathology.

In [ ]:
from google.colab import files
import kagglehub

dataset_raw_path = "./skin_disease_dataset"
os.makedirs(dataset_raw_path, exist_ok=True)

# Authenticate with kaggle.json if present, or prompt upload
if not os.path.exists('kaggle.json'):
    print("🔑 Upload your kaggle.json file (if you have one) or cancel to try direct download:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('kaggle.json'):
            shutil.move(fn, 'kaggle.json')

if os.path.exists('kaggle.json'):
    with open('kaggle.json') as f:
        creds = json.load(f)
    os.environ['KAGGLE_USERNAME'] = creds.get('username', '')
    os.environ['KAGGLE_KEY'] = creds.get('key', '')
    print(f"✅ Authenticated as: {creds.get('username', 'user')}")
    print("📥 Downloading HAM10000 dataset via Kaggle CLI...")
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 --unzip -p {dataset_raw_path}
else:
    print("📥 Downloading HAM10000 dataset via KaggleHub...")
    try:
        downloaded_dir = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
        dataset_raw_path = downloaded_dir
    except Exception as e:
        print(f"KaggleHub fallback: {e}")

print(f"\n📁 Dataset root: {dataset_raw_path}")
for root, dirs, fnames in os.walk(dataset_raw_path):
    depth = root[len(dataset_raw_path):].count(os.sep)
    if depth < 2:
        print(f"{'  ' * depth}📁 {os.path.basename(root)}/ ({len(fnames)} files)")

---
## 3. Data Ingestion & Clinical Label Standardization
We standardize categories to the 7 clinical diagnostic classes defined in Arogya Sahayak's `scanConfig.ts`:
- `0: Melanocytic Nevus (Benign Mole)`
- `1: Melanoma (Malignant Neoplasm)`
- `2: Benign Keratosis (Seborrheic Plaque)`
- `3: Basal Cell Carcinoma`
- `4: Actinic Keratosis (Pre-Cancerous)`
- `5: Vascular Lesion (Angioma)`
- `6: Dermatofibroma (Benign)`

In [ ]:
CLASS_NAMES = [
    'Melanocytic Nevus (Benign Mole)',
    'Melanoma (Malignant Neoplasm)',
    'Benign Keratosis (Seborrheic Plaque)',
    'Basal Cell Carcinoma',
    'Actinic Keratosis (Pre-Cancerous)',
    'Vascular Lesion (Angioma)',
    'Dermatofibroma (Benign)'
]

DX_TO_IDX = {
    'nv': 0,
    'mel': 1,
    'bkl': 2,
    'bcc': 3,
    'akiec': 4,
    'vasc': 5,
    'df': 6
}

# Search for HAM10000_metadata.csv
metadata_csv_path = None
for root, dirs, fnames in os.walk(dataset_raw_path):
    for fn in fnames:
        if fn.lower() == 'ham10000_metadata.csv':
            metadata_csv_path = os.path.join(root, fn)
            break
    if metadata_csv_path:
        break

# Map image filenames to disk paths
print("Indexing image files on disk...")
image_path_dict = {}
for ext in ('*.jpg', '*.jpeg', '*.png'):
    for p in glob.glob(os.path.join(dataset_raw_path, "**", ext), recursive=True):
        base_id = os.path.splitext(os.path.basename(p))[0]
        image_path_dict[base_id] = p
print(f"Indexed {len(image_path_dict):,} images on disk.")

records = []
if metadata_csv_path and os.path.exists(metadata_csv_path):
    print(f"Found metadata: {metadata_csv_path}")
    meta_df = pd.read_csv(metadata_csv_path)
    for _, row in meta_df.iterrows():
        img_id = str(row['image_id']).strip()
        dx = str(row['dx']).lower().strip()
        if img_id in image_path_dict and dx in DX_TO_IDX:
            records.append({
                'filepath': image_path_dict[img_id],
                'label_idx': DX_TO_IDX[dx],
                'label_name': CLASS_NAMES[DX_TO_IDX[dx]]
            })
else:
    # Fallback to directory name parsing if pre-organized into class folders
    print("Metadata CSV not found; parsing class directory structure...")
    for base_id, p in image_path_dict.items():
        parent = os.path.basename(os.path.dirname(p)).lower().strip()
        for dx, idx in DX_TO_IDX.items():
            if dx in parent:
                records.append({
                    'filepath': p,
                    'label_idx': idx,
                    'label_name': CLASS_NAMES[idx]
                })
                break

df_all = pd.DataFrame(records)
print(f"\nTotal Dermoscopic Lesions Ingested: {len(df_all):,}")
print("\nClass Distribution across Dataset:")
print(df_all['label_name'].value_counts())

---
## 4. Stratified Dataset Partitioning (70% Train, 15% Val, 15% Test)
Preserves proportional clinical distribution across all 7 diagnostic classes.

In [ ]:
df_train, df_temp = train_test_split(
    df_all,
    test_size=0.30,
    random_state=SEED,
    stratify=df_all['label_idx'].values
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=df_temp['label_idx'].values
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print(f"Training Cohort:   {len(df_train):,} images ({len(df_train)/len(df_all):.1%})")
print(f"Validation Cohort: {len(df_val):,} images ({len(df_val)/len(df_all):.1%})")
print(f"Test Cohort:       {len(df_test):,} images ({len(df_test)/len(df_all):.1%})")

---
## 5. Visualizing Representative Dermoscopic Lesions
Examine clinical morphological variations across moles, melanomas, keratoses, and vascular clods.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
viz_samples = []
sample_indices = [0, 1, 2, 3]  # Nevus, Melanoma, Keratosis, BCC
for c_idx in sample_indices:
    subset = df_train[df_train['label_idx'] == c_idx]
    if len(subset) > 0:
        viz_samples.append(subset.sample(1, random_state=SEED).iloc[0])
    else:
        viz_samples.append(df_train.iloc[c_idx])

for i, row in enumerate(viz_samples):
    img = Image.open(row['filepath']).convert('RGB')
    axes[i].imshow(img)
    axes[i].set_title(f"{row['label_name']}", fontsize=9.5, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('Representative HAM10000 Dermoscopic Lesions across Pathological Classes', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Input Pipeline & Dermoscopic Data Augmentation
Standardize to $224 \times 224 \times 3$ with MobileNetV2 $[-1, 1]$ normalization.  
Dermoscopy images are circular/planar without fixed orientation; horizontal + vertical flips, rotations ($\pm 30^\circ$), zoom ($\pm 5\%$), and contrast jitter simulate clinical lighting.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 7

skin_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.08, fill_mode='nearest'),
    layers.RandomZoom((-0.05, 0.05), fill_mode='nearest'),
    layers.RandomContrast(0.06),
], name="skin_augmentation")

def load_and_preprocess_skin(path, label):
    img_raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_raw, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = preprocess_input(img)  # MobileNetV2 [-1, 1] scaling
    one_hot = tf.one_hot(label, depth=NUM_CLASSES)
    return img, one_hot

def create_skin_dataset(df, is_training=False):
    paths = df['filepath'].values
    labels = df['label_idx'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess_skin, num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        ds = ds.shuffle(buffer_size=1024, seed=SEED)
        ds = ds.map(lambda x, y: (skin_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_skin_dataset(df_train, is_training=True)
val_ds   = create_skin_dataset(df_val, is_training=False)
test_ds  = create_skin_dataset(df_test, is_training=False)

# Class weight calculation to counter severe imbalance (Nevus is ~67% of cases)
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df_train['label_idx']),
    y=df_train['label_idx'].values
)
class_weights_dict = {i: float(w) for i, w in enumerate(weights)}
print("Computed Balanced Class Weights:")
for k, v in class_weights_dict.items():
    print(f"  Class {k} ({CLASS_NAMES[k]}): weight = {v:.3f}")

---
## 7. Model Architecture (MobileNetV2 Transfer Learning)
We use MobileNetV2 with Global Average Pooling, Batch Normalization, Dense(128, ReLU), Dropout (0.35), and a 7-class Softmax classification head.

In [ ]:
def build_skin_model():
    inputs = keras.Input(shape=(224, 224, 3), name="input_dermoscopy")
    
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_tensor=inputs
    )
    base_model.trainable = False  # Freeze for Phase 1
    
    x = base_model.output
    x = layers.GlobalAveragePooling2D(name="skin_global_pool")(x)
    x = layers.BatchNormalization(name="skin_bn")(x)
    x = layers.Dense(128, activation='relu', name="skin_dense_128")(x)
    x = layers.Dropout(0.35, name="skin_dropout")(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name="skin_probabilities")(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs, name="Arogya_SkinDisease_MobileNetV2")
    return model, base_model

model, base_model = build_skin_model()
model.summary()

---
## 8. Phase 1 Training: Feature Adaptation (Frozen Backbone)
Train custom dense head with Adam ($10^{-3}$) and categorical crossentropy loss.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

callbacks_phase1 = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6, verbose=1)
]

print("Starting Phase 1 Training (Frozen Backbone)...")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=6,
    class_weight=class_weights_dict,
    callbacks=callbacks_phase1,
    verbose=1
)

---
## 9. Phase 2 Training: Fine-Tuning Upper Convolutional Layers
Unfreeze the top 35 layers of MobileNetV2 (including bottleneck block 16 and final conv `Conv_1`) and fine-tune with a low learning rate ($10^{-5}$) to adapt filters to pigment networks, atypical borders, and regression structures.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-35]:
    layer.trainable = False

trainable_params = sum(np.prod(p.shape) for p in model.trainable_weights)
print(f"Trainable parameters for fine-tuning: {trainable_params:,}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

callbacks_phase2 = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1)
]

print("Starting Phase 2 Fine-Tuning...")
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    class_weight=class_weights_dict,
    callbacks=callbacks_phase2,
    verbose=1
)

In [ ]:
acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy', color='#10b981', lw=2)
plt.plot(val_acc, label='Validation Accuracy', color='#3b82f6', lw=2)
plt.axvline(x=len(history_phase1.history['accuracy'])-1, color='gray', linestyle='--', label='Fine-Tuning Start')
plt.title('Skin Lesion Classifier: Accuracy Trajectory', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss', color='#ef4444', lw=2)
plt.plot(val_loss, label='Validation Loss', color='#f59e0b', lw=2)
plt.axvline(x=len(history_phase1.history['loss'])-1, color='gray', linestyle='--', label='Fine-Tuning Start')
plt.title('Skin Lesion Classifier: Loss Trajectory', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Categorical Crossentropy')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Clinical Evaluation on Held-Out Test Cohort
Assess diagnostic sensitivity (Melanoma & Carcinoma Recall), specificity, precision, and F1-score across untouched test dermoscopic scans.

In [ ]:
print("Evaluating on untouched Test Set...")
y_true = []
y_pred_probs = []

for batch_imgs, batch_labels in test_ds:
    probs = model.predict(batch_imgs, verbose=0)
    y_true.extend(np.argmax(batch_labels.numpy(), axis=1))
    y_pred_probs.extend(probs)

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification report
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, output_dict=True)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix: Held-Out HAM10000 Test Cohort', fontweight='bold')
plt.xlabel('Predicted Diagnosis')
plt.ylabel('Ground Truth')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

---
## 11. Dermal Lesion Grad-CAM Saliency Localization
Extract partial derivatives from the final convolutional layer (`Conv_1`) to verify that the network focuses on pigment networks and irregular lesion contours.

In [ ]:
def make_skin_gradcam_heatmap(img_array, model, last_conv_layer_name="Conv_1", pred_index=None):
    grad_model = keras.models.Model(
        inputs=[model.inputs],
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

# Visualize Grad-CAM on 4 representative test samples
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
viz_test_rows = []
for c_idx in [0, 1, 2, 3]:
    class_subset = df_test[df_test['label_idx'] == c_idx]
    if len(class_subset) > 0:
        viz_test_rows.append(class_subset.iloc[0])
    else:
        viz_test_rows.append(df_test.iloc[c_idx])
viz_test = pd.DataFrame(viz_test_rows).reset_index(drop=True)

for idx in range(4):
    sample_path = viz_test.iloc[idx]['filepath']
    true_label = viz_test.iloc[idx]['label_name']
    
    img_pil = Image.open(sample_path).convert('RGB').resize(IMG_SIZE)
    img_arr = np.array(img_pil)
    img_tensor = preprocess_input(np.expand_dims(img_arr.astype(np.float32), axis=0))
    
    pred = model.predict(img_tensor, verbose=0)[0]
    pred_idx = np.argmax(pred)
    heatmap = make_skin_gradcam_heatmap(img_tensor, model)
    
    axes[0, idx].imshow(img_pil)
    axes[0, idx].set_title(f"True: {true_label[:20]}...\nPred: {CLASS_NAMES[pred_idx][:20]}... ({pred[pred_idx]:.1%})", fontsize=9.5)
    axes[0, idx].axis('off')
    
    heatmap_resized = np.uint8(255 * heatmap)
    axes[1, idx].imshow(img_pil)
    axes[1, idx].imshow(heatmap_resized, cmap='jet', alpha=0.45)
    axes[1, idx].set_title(f"Conv_1 Grad-CAM Localization", fontsize=9.5, fontweight='bold', color='navy')
    axes[1, idx].axis('off')

plt.suptitle('Clinical Grad-CAM Verification on Dermoscopic Lesions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 12. TensorFlow.js Web Export (Direct In-Memory Exporter)
Export Keras model directly into TensorFlow.js format (`model.json` + `group1-shard1of1.bin`, ~9.5 MB) using pure Python memory buffers.

In [ ]:
EXPORT_DIR = "./arogya_skin_disease_tfjs/web_model"
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Extract Keras topology
try:
    topology = json.loads(model.to_json())
except Exception as e:
    print(f"Topology warning: {e}")
    topology = {}

# 2. Extract weights into binary shard
manifest_weights = []
raw_bytes = bytearray()
for w in model.weights:
    w_name = w.path if hasattr(w, 'path') else w.name
    if isinstance(w_name, str) and w_name.endswith(':0'):
        w_name = w_name[:-2]
    w_arr = w.numpy().astype(np.float32)
    manifest_weights.append({
        "name": str(w_name),
        "shape": list(w_arr.shape),
        "dtype": "float32"
    })
    raw_bytes.extend(w_arr.tobytes())

shard_filename = "group1-shard1of1.bin"
shard_path = os.path.join(EXPORT_DIR, shard_filename)
with open(shard_path, "wb") as f:
    f.write(raw_bytes)
print(f"  ✅ Saved weight shard: {shard_filename} ({len(raw_bytes) / (1024*1024):.2f} MB)")

# 3. Write model.json
tfjs_manifest = {
    "format": "layers-model",
    "generatedBy": f"keras v{tf.__version__}",
    "convertedBy": "Arogya Sahayak Direct Exporter",
    "modelTopology": topology,
    "weightsManifest": [{
        "paths": [shard_filename],
        "weights": manifest_weights
    }]
}
model_json_path = os.path.join(EXPORT_DIR, "model.json")
with open(model_json_path, 'w', encoding='utf-8') as f:
    json.dump(tfjs_manifest, f, indent=2)
print("  ✅ Saved model.json topology.")

---
## 13. Model Card & Clinical Audit Metadata
Generate Mitchell et al. `model_card.json` and `metrics.json` recording sensitivity, specificity, and ethical safety boundaries.

In [ ]:
import datetime

class_metrics = []
for c_idx, c_name in enumerate(CLASS_NAMES):
    sens = float(cm[c_idx, c_idx] / cm[c_idx].sum()) if cm[c_idx].sum() > 0 else 0.0
    class_metrics.append({
        "index": c_idx,
        "label": c_name,
        "sensitivity": sens,
        "precision": float(report[c_name]['precision']),
        "f1_score": float(report[c_name]['f1-score'])
    })

overall_accuracy = float(np.sum(np.diag(cm)) / np.sum(cm))

model_card = {
    "schema_version": "1.0.0",
    "model_name": "Arogya Sahayak Skin Disease HAM10000 MobileNetV2 Classifier",
    "scanner_id": "skin_disease",
    "modality": "Skin Dermoscopy Images",
    "input_dimensions": [224, 224, 3],
    "preprocessing": "MobileNetV2 scale [-1, 1]",
    "classes": class_metrics,
    "metrics": {
        "overall_test_accuracy": float(overall_accuracy),
        "macro_avg_f1": float(report['macro avg']['f1-score']),
        "weighted_avg_f1": float(report['weighted avg']['f1-score'])
    },
    "calibrated_threshold": 0.52,
    "last_conv_layer": "Conv_1",
    "training_metadata": {
        "date": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "institution": "ABIT Cuttack / BPUT Odisha",
        "developers": ["Rahul Prasad Das", "Purnachandra Swain", "Rohit Kumar Thakur", "Ritik Kumar Swain"],
        "guide": "Prof. Rajesh Kumar Sahoo",
        "dataset_source": "ISIC HAM10000 Benchmark (Kaggle: kmader/skin-cancer-mnist-ham10000)",
        "training_samples": len(df_train),
        "validation_samples": len(df_val),
        "test_samples": len(df_test),
        "hardware": "Google Colab Tesla T4 GPU (Free Tier)"
    },
    "clinical_safety_notice": "SCREENING AID ONLY. Not intended as a definitive diagnostic tool for malignant melanoma or carcinomas. Confirmatory diagnosis requires in-person polarized dermoscopy, ABCDE clinical rule verification, and punch / excisional biopsy with histopathological examination by a board-certified dermatologist."
}

model_card_file = os.path.join(EXPORT_DIR, "model_card.json")
with open(model_card_file, 'w', encoding='utf-8') as f:
    json.dump(model_card, f, indent=2)

metrics_file = os.path.join(EXPORT_DIR, "metrics.json")
with open(metrics_file, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

print("Model Card & Audit Metrics JSON generated successfully.")
print(json.dumps(model_card, indent=2))

---
## 14. Package & Download Model for Arogya Sahayak Integration
Packages the exported web model into `arogyasahayak_skin_disease_model.zip` and triggers an automatic browser download.

In [ ]:
import zipfile
from google.colab import files

zip_filename = "arogyasahayak_skin_disease_model.zip"
print(f"Creating deployment archive: {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, filenames in os.walk(EXPORT_DIR):
        for fname in filenames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, EXPORT_DIR)
            zipf.write(fpath, arcname)
            print(f"  Added: {arcname}")

zip_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"\nPackage ready! Size: {zip_size_mb:.2f} MB")
print("Initiating browser download to your local machine...")
files.download(zip_filename)

print("""
==========================================================================
✅ DEPLOYMENT INSTRUCTIONS:
1. Extract 'arogyasahayak_skin_disease_model.zip'.
2. Copy the extracted files into your Arogya Sahayak repository at:
   public/models/skin_disease/
   (Files needed: model.json, group1-shard1of1.bin, model_card.json, metrics.json)
3. Done! Arogya Sahayak will now run real deep-learning Skin Disease screening!
==========================================================================
""")